# Why Booking Behaviour Matters

Understanding customer booking behaviour helps hotels make better business decisions by identifying patterns in reservations and cancellations. By analysing how guests book rooms, how long they stay, and when they are most likely to cancel, hotels can improve occupancy rates, revenue management, staffing plans, pricing strategies, and customer service. Insights from booking behaviour also help hotels reduce revenue losses caused by cancellations and develop targeted marketing campaigns for different customer segments.

# Business Questions

### 1. Which hotel type is booked most frequently?
### 2. Does the length of stay influence cancellation behaviour?
### 3. Does lead time affect the cancellation rate?

# Project Objective

The primary objective of this project is to analyse hotel booking data from 2017–2019 and create meaningful visualisations that explain customer booking and cancellation behaviour. The final deliverable is a data-driven report containing charts, insights, and business recommendations related to hotel type popularity, stay duration, and lead time effects on cancellations.

# Dataset Overview

The dataset contains hotel booking records from 2017 to 2019 with approximately 119,000 observations and 29 variables. It includes information about hotel type, booking dates, lead time, length of stay, cancellation status, guest details, room reservations, pricing, and other booking-related attributes. The dataset provides sufficient information to analyse customer booking patterns, compare hotel types, evaluate cancellation behaviour, and generate business recommendations based on visual evidence.




In [894]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

In [895]:
import pandas as pd

df = pd.read_csv("Data/hotel_bookings_data.csv")

print("Dataset loaded successfully!")
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

Dataset loaded successfully!
Rows: 119390
Columns: 29


In [896]:
df.head()

,hotel,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_weekdays_nights,adults,children,babies,meal,city,market_segment,distribution_channel,is_repeated_guest,previous_cancellations,previous_bookings_not_canceled,booking_changes,deposit_type,agent,company,days_in_waiting_list,customer_type,adr,required_car_parking_spaces,total_of_special_requests,reservation_status
0,Resort Hotel,0,342,2017,September,27,1,0,0,2,0.0,0,Breakfast,Kota Denpasar,Direct,Direct,0,0,0,3,No Deposit,NaN,NaN,0,Personal,0.0,0,0,Check-Out
1,Resort Hotel,0,737,2017,September,27,1,0,0,2,0.0,0,Breakfast,Kota Denpasar,Direct,Direct,0,0,0,4,No Deposit,NaN,NaN,0,Personal,0.0,0,0,Check-Out
2,Resort Hotel,0,7,2017,September,27,1,0,1,1,0.0,0,Breakfast,Kabupaten Bangka,Direct,Direct,0,0,0,0,No Deposit,NaN,NaN,0,Personal,75.0,0,0,Check-Out
3,Resort Hotel,0,13,2017,September,27,1,0,1,1,0.0,0,Breakfast,Kabupaten Bangka,Corporate,Corporate,0,0,0,0,No Deposit,304.0,NaN,0,Personal,75.0,0,0,Check-Out
4,Resort Hotel,0,14,2017,September,27,1,0,2,2,0.0,0,Breakfast,Kabupaten Bangka,Online TA,TA/TO,0,0,0,0,No Deposit,240.0,NaN,0,Personal,98.0,0,1,Check-Out


In [897]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 119390 entries, 0 to 119389
Data columns (total 29 columns):
 #   Column                          Non-Null Count   Dtype  
---  ------                          --------------   -----  
 0   hotel                           119390 non-null  str    
 1   is_canceled                     119390 non-null  int64  
 2   lead_time                       119390 non-null  int64  
 3   arrival_date_year               119390 non-null  int64  
 4   arrival_date_month              119390 non-null  str    
 5   arrival_date_week_number        119390 non-null  int64  
 6   arrival_date_day_of_month       119390 non-null  int64  
 7   stays_in_weekend_nights         119390 non-null  int64  
 8   stays_in_weekdays_nights        119390 non-null  int64  
 9   adults                          119390 non-null  int64  
 10  children                        119386 non-null  float64
 11  babies                          119390 non-null  int64  
 12  meal                       

In [898]:
print("Columns in the dataset:")
for column in df.columns:
    print(column)

Columns in the dataset:
hotel
is_canceled
lead_time
arrival_date_year
arrival_date_month
arrival_date_week_number
arrival_date_day_of_month
stays_in_weekend_nights
stays_in_weekdays_nights
adults
children
babies
meal
city
market_segment
distribution_channel
is_repeated_guest
previous_cancellations
previous_bookings_not_canceled
booking_changes
deposit_type
agent
company
days_in_waiting_list
customer_type
adr
required_car_parking_spaces
total_of_special_requests
reservation_status


In [899]:
missing_values = df.isnull().sum()

missing_values = missing_values[
    missing_values > 0
].sort_values(ascending=False)

missing_values

company     112593
agent        16340
city           488
children         4
dtype: int64

In [900]:
print("Duplicate rows:", df.duplicated().sum())

Duplicate rows: 33261


In [901]:
df = df.drop_duplicates().copy()

print("Dataset after removing duplicates:")
print(df.shape)

Dataset after removing duplicates:
(86129, 29)


In [902]:
df["children"] = df["children"].fillna(0)
df["agent"] = df["agent"].fillna(0)
df["company"] = df["company"].fillna(0)
df["city"] = df["city"].fillna("Unknown")

print("Missing values handled.")

Missing values handled.


In [903]:
df["total_guests"] = (
    df["adults"] +
    df["children"] +
    df["babies"]
)

df = df[df["total_guests"] > 0].copy()

print("Rows after removing invalid guest records:", len(df))

Rows after removing invalid guest records: 85964


In [904]:
df["total_stay"] = (
    df["stays_in_weekend_nights"] +
    df["stays_in_weekdays_nights"]
)

df[[
    "stays_in_weekend_nights",
    "stays_in_weekdays_nights",
    "total_stay"
]].head()

,stays_in_weekend_nights,stays_in_weekdays_nights,total_stay
0,0,0,0
1,0,0,0
2,0,1,1
3,0,1,1
4,0,2,2


In [905]:
df["cancellation_status"] = df["is_canceled"].map({
    0: "Not Cancelled",
    1: "Cancelled"
})

df[
    ["is_canceled", "cancellation_status"]
].head()

,is_canceled,cancellation_status
0,0,Not Cancelled
1,0,Not Cancelled
2,0,Not Cancelled
3,0,Not Cancelled
4,0,Not Cancelled


In [906]:
df.describe()

,is_canceled,lead_time,arrival_date_year,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_weekdays_nights,adults,children,babies,is_repeated_guest,previous_cancellations,previous_bookings_not_canceled,booking_changes,agent,company,days_in_waiting_list,adr,required_car_parking_spaces,total_of_special_requests,total_guests,total_stay
count,85964.000000,85964.000000,85964.000000,85964.000000,85964.000000,85964.000000,85964.000000,85964.000000,85964.000000,85964.000000,85964.000000,85964.000000,85964.000000,85964.000000,85964.000000,85964.000000,85964.000000,85964.000000,85964.000000,85964.000000,85964.000000,85964.000000
mean,0.275929,79.324880,2018.215532,26.793111,15.806908,1.009085,2.631718,1.882160,0.140547,0.011005,0.038993,0.030559,0.186636,0.268833,81.284445,10.767147,0.641583,106.816889,0.085315,0.705435,2.033712,3.640803
std,0.446984,85.504731,0.683901,13.669615,8.840868,1.028835,2.044414,0.622734,0.458736,0.114530,0.193580,0.371669,1.745567,0.712725,109.995675,53.389781,9.265003,55.051347,0.283197,0.833630,0.792138,2.749521
min,0.000000,0.000000,2017.000000,1.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,-6.380000,0.000000,0.000000,1.000000,0.000000
25%,0.000000,11.000000,2018.000000,16.000000,8.000000,0.000000,1.000000,2.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,9.000000,0.000000,0.000000,72.250000,0.000000,0.000000,2.000000,2.000000
50%,0.000000,49.000000,2018.000000,27.000000,16.000000,1.000000,2.000000,2.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,9.000000,0.000000,0.000000,99.000000,0.000000,1.000000,2.000000,3.000000
75%,1.000000,124.000000,2019.000000,37.000000,23.000000,2.000000,4.000000,2.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,240.000000,0.000000,0.000000,134.800000,0.000000,1.000000,2.000000,5.000000
max,1.000000,737.000000,2019.000000,53.000000,31.000000,19.000000,50.000000,55.000000,10.000000,10.000000,1.000000,26.000000,72.000000,18.000000,535.000000,543.000000,391.000000,5400.000000,8.000000,5.000000,55.000000,69.000000


In [907]:
hotel_bookings = (
    df["hotel"]
    .value_counts()
    .reset_index()
)

hotel_bookings.columns = [
    "Hotel",
    "Bookings"
]

hotel_bookings

,Hotel,Bookings
0,City Hotel,52422
1,Resort Hotel,33542


In [908]:
hotel_bookings["Percentage"] = (
    hotel_bookings["Bookings"] /
    hotel_bookings["Bookings"].sum() * 100
)

hotel_bookings

,Hotel,Bookings,Percentage
0,City Hotel,52422,60.981341
1,Resort Hotel,33542,39.018659


In [909]:
fig = px.bar(
    hotel_bookings,
    x="Hotel",
    y="Bookings",
    text="Bookings",
    title="Bookings by Hotel Type"
)

fig.update_traces(
    textposition="outside"
)

fig.show()

In [910]:
fig = px.pie(
    hotel_bookings,
    names="Hotel",
    values="Bookings",
    hole=0.55,
    title="Hotel Booking Share"
)

fig.show()

In [911]:
month_order = [
    "January",
    "February",
    "March",
    "April",
    "May",
    "June",
    "July",
    "August",
    "September",
    "October",
    "November",
    "December"
]

df["arrival_date_month"] = pd.Categorical(
    df["arrival_date_month"],
    categories=month_order,
    ordered=True
)

monthly_bookings = (
    df.groupby(
        "arrival_date_month",
        observed=True
    )
    .size()
    .reset_index(name="Bookings")
)

monthly_bookings

,arrival_date_month,Bookings
0,January,4901
1,February,5063
2,March,4624
3,April,6020
4,May,7411
5,June,7803
6,July,8231
7,August,7648
8,September,9973
9,October,11135


In [912]:
fig = px.line(
    monthly_bookings,
    x="arrival_date_month",
    y="Bookings",
    markers=True,
    title="Monthly Booking Trend"
)

fig.update_layout(
    xaxis_title="Month",
    yaxis_title="Number of Bookings"
)

fig.show()

In [913]:
monthly_hotel = (
    df.groupby(
        ["arrival_date_month", "hotel"],
        observed=True
    )
    .size()
    .reset_index(name="Bookings")
)

fig = px.line(
    monthly_hotel,
    x="arrival_date_month",
    y="Bookings",
    color="hotel",
    markers=True,
    title="Monthly Bookings by Hotel Type"
)

fig.show()

In [914]:
cancellation_summary = (
    df["cancellation_status"]
    .value_counts()
    .reset_index()
)

cancellation_summary.columns = [
    "Status",
    "Bookings"
]

cancellation_summary

,Status,Bookings
0,Not Cancelled,62244
1,Cancelled,23720


In [915]:
cancellation_summary["Percentage"] = (
    cancellation_summary["Bookings"] /
    cancellation_summary["Bookings"].sum() * 100
)

cancellation_summary

,Status,Bookings,Percentage
0,Not Cancelled,62244,72.407054
1,Cancelled,23720,27.592946


In [916]:
fig = px.pie(
    cancellation_summary,
    names="Status",
    values="Bookings",
    hole=0.55,
    title="Cancelled vs Non-Cancelled Bookings"
)

fig.show()

In [917]:
hotel_cancellation = (
    df.groupby("hotel")["is_canceled"]
    .mean()
    .reset_index()
)

hotel_cancellation["Cancellation Rate"] = (
    hotel_cancellation["is_canceled"] * 100
)

hotel_cancellation

,hotel,is_canceled,Cancellation Rate
0,City Hotel,0.302163,30.216321
1,Resort Hotel,0.234929,23.492934


In [918]:
fig = px.bar(
    hotel_cancellation,
    x="hotel",
    y="Cancellation Rate",
    text="Cancellation Rate",
    title="Cancellation Rate by Hotel Type"
)

fig.update_traces(
    texttemplate="%{text:.1f}%",
    textposition="outside"
)

fig.show()

In [919]:
stay_cancellation = (
    df.groupby("total_stay")["is_canceled"]
    .mean()
    .reset_index()
)

stay_cancellation["Cancellation Rate"] = (
    stay_cancellation["is_canceled"] * 100
)

stay_cancellation = stay_cancellation[
    stay_cancellation["total_stay"] <= 30
]

stay_cancellation.head()

,total_stay,is_canceled,Cancellation Rate
0,0,0.039249,3.924915
1,1,0.184055,18.405489
2,2,0.269306,26.930603
3,3,0.300337,30.033662
4,4,0.306606,30.660589


In [920]:
fig = px.line(
    stay_cancellation,
    x="total_stay",
    y="Cancellation Rate",
    markers=True,
    title="Cancellation Rate vs Length of Stay"
)

fig.update_layout(
    xaxis_title="Length of Stay (Nights)",
    yaxis_title="Cancellation Rate (%)"
)

fig.show()

In [921]:
lead_time_summary = (
    df.groupby("lead_time")["is_canceled"]
    .mean()
    .reset_index()
)

lead_time_summary["Cancellation Rate"] = (
    lead_time_summary["is_canceled"] * 100
)

lead_time_summary.head()

,lead_time,is_canceled,Cancellation Rate
0,0,0.058261,5.826072
1,1,0.066206,6.620646
2,2,0.094566,9.456635
3,3,0.099823,9.982280
4,4,0.098065,9.806452


In [922]:
bins = [
    -1,
    7,
    30,
    60,
    90,
    120,
    180,
    365,
    1000
]

labels = [
    "0–7 Days",
    "8–30 Days",
    "31–60 Days",
    "61–90 Days",
    "91–120 Days",
    "121–180 Days",
    "181–365 Days",
    "365+ Days"
]

df["lead_time_group"] = pd.cut(
    df["lead_time"],
    bins=bins,
    labels=labels
)

In [923]:
lead_analysis = (
    df.groupby(
        "lead_time_group",
        observed=True
    )["is_canceled"]
    .mean()
    .reset_index()
)

lead_analysis["Cancellation Rate"] = (
    lead_analysis["is_canceled"] * 100
)

lead_analysis

,lead_time_group,is_canceled,Cancellation Rate
0,0–7 Days,0.084201,8.420062
1,8–30 Days,0.254186,25.418597
2,31–60 Days,0.317812,31.781176
3,61–90 Days,0.327227,32.722740
4,91–120 Days,0.349184,34.918402
5,121–180 Days,0.351162,35.116193
6,181–365 Days,0.402595,40.259501
7,365+ Days,0.425047,42.504744


In [924]:
fig = px.bar(
    lead_analysis,
    x="lead_time_group",
    y="Cancellation Rate",
    text="Cancellation Rate",
    title="Cancellation Rate by Lead Time"
)

fig.update_traces(
    texttemplate="%{text:.1f}%",
    textposition="outside"
)

fig.update_layout(
    xaxis_title="Lead Time",
    yaxis_title="Cancellation Rate (%)"
)

fig.show()

In [925]:
fig = px.histogram(
    df,
    x="lead_time",
    color="hotel",
    nbins=50,
    title="Distribution of Booking Lead Time"
)

fig.update_layout(
    xaxis_title="Lead Time (Days)",
    yaxis_title="Number of Bookings"
)

fig.show()

In [926]:
adr_by_hotel = (
    df.groupby("hotel")["adr"]
    .mean()
    .reset_index()
)

adr_by_hotel.columns = [
    "Hotel",
    "Average ADR"
]

adr_by_hotel

,Hotel,Average ADR
0,City Hotel,111.561255
1,Resort Hotel,99.402030


In [927]:
fig = px.bar(
    adr_by_hotel,
    x="Hotel",
    y="Average ADR",
    text="Average ADR",
    title="Average Daily Rate by Hotel"
)

fig.update_traces(
    texttemplate="%{text:.2f}",
    textposition="outside"
)

fig.show()

In [928]:
segment_analysis = (
    df["market_segment"]
    .value_counts()
    .reset_index()
)

segment_analysis.columns = [
    "Market Segment",
    "Bookings"
]

segment_analysis

,Market Segment,Bookings
0,Online TA,51328
1,Offline TA/TO,13458
2,Direct,11716
3,Groups,4438
4,Corporate,4115
5,Complementary,681
6,Aviation,226
7,Undefined,2


In [929]:
fig = px.bar(
    segment_analysis,
    x="Market Segment",
    y="Bookings",
    text="Bookings",
    title="Bookings by Market Segment"
)

fig.update_traces(
    textposition="outside"
)

fig.show()

In [930]:
fig = px.bar(
    segment_analysis,
    x="Market Segment",
    y="Bookings",
    text="Bookings",
    title="Bookings by Market Segment"
)

fig.update_traces(
    textposition="outside"
)

fig.show()

In [931]:
total_bookings = len(df)

cancelled_bookings = df["is_canceled"].sum()

cancellation_rate = (
    cancelled_bookings /
    total_bookings * 100
)

average_lead_time = df["lead_time"].mean()

average_stay = df["total_stay"].mean()

average_adr = df["adr"].mean()

print("========== HOTEL KPIs ==========")
print(f"Total Bookings       : {total_bookings:,}")
print(f"Cancelled Bookings   : {cancelled_bookings:,}")
print(f"Cancellation Rate    : {cancellation_rate:.2f}%")
print(f"Average Lead Time    : {average_lead_time:.2f} days")
print(f"Average Stay         : {average_stay:.2f} nights")
print(f"Average ADR          : {average_adr:.2f}")

========== HOTEL KPIs ==========
Total Bookings       : 85,964
Cancelled Bookings   : 23,720
Cancellation Rate    : 27.59%
Average Lead Time    : 79.32 days
Average Stay         : 3.64 nights
Average ADR          : 106.82


In [932]:
peak_month = monthly_bookings.loc[
    monthly_bookings["Bookings"].idxmax()
]

lowest_month = monthly_bookings.loc[
    monthly_bookings["Bookings"].idxmin()
]

print(
    f"Peak booking month: "
    f"{peak_month['arrival_date_month']} "
    f"({peak_month['Bookings']:,} bookings)"
)

print(
    f"Lowest booking month: "
    f"{lowest_month['arrival_date_month']} "
    f"({lowest_month['Bookings']:,} bookings)"
)

Peak booking month: October (11,135 bookings)
Lowest booking month: March (4,624 bookings)


In [933]:
print("========== BUSINESS RECOMMENDATIONS ==========\n")

print("1. PEAK SEASON")
print(
    f"Prepare additional capacity and marketing campaigns "
    f"before {peak_month['arrival_date_month']}."
)

print("\n2. CANCELLATION MANAGEMENT")
print(
    f"The overall cancellation rate is "
    f"{cancellation_rate:.1f}%. "
    f"Hotels should consider deposits, reminder messages "
    f"and flexible rescheduling."
)

print("\n3. LEAD TIME")
print(
    f"The average booking lead time is "
    f"{average_lead_time:.0f} days. "
    f"Advance reservations should receive confirmation "
    f"and reminder communication."
)

print("\n4. STAY DURATION")
print(
    f"The average customer stays approximately "
    f"{average_stay:.1f} nights. "
    f"Hotels can design packages around common stay lengths."
)

print("\n5. HOTEL TYPE")
print(
    f"The most frequently booked hotel type is "
    f"{hotel_bookings.iloc[0]['Hotel']}."
)

========== BUSINESS RECOMMENDATIONS ==========

1. PEAK SEASON
Prepare additional capacity and marketing campaigns before October.

2. CANCELLATION MANAGEMENT
The overall cancellation rate is 27.6%. Hotels should consider deposits, reminder messages and flexible rescheduling.

3. LEAD TIME
The average booking lead time is 79 days. Advance reservations should receive confirmation and reminder communication.

4. STAY DURATION
The average customer stays approximately 3.6 nights. Hotels can design packages around common stay lengths.

5. HOTEL TYPE
The most frequently booked hotel type is City Hotel.
